# Day 23：FastAPI 后端服务开发 Part 1 (`v0.2.0-day23`)

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》Day 23 实战 Notebook！今天我们将使用 `fastapi.testclient.TestClient` 在进程内零端口冲突地测试 `project_iterations/day_23` 的全部 REST API 端点、Path/Query 参数与 Pydantic v2 校验规则。

## 1. 导入 Day 23 FastAPI 应用并初始化 `TestClient`
通过 `TestClient(app)`，无需启动后台 Uvicorn 进程即可完整模拟 HTTP 请求与响应。

In [1]:
import json
import sys
from pathlib import Path
from fastapi.testclient import TestClient

for candidate in [Path.cwd(), *Path.cwd().parents]:
    iter_dir = candidate / 'project_iterations' / 'day_23'
    if iter_dir.exists():
        if str(iter_dir) not in sys.path:
            sys.path.insert(0, str(iter_dir))
        break

from zhishu_agent.api_server import APP_VERSION, app

client = TestClient(app)
print(f'已加载 FastAPI 应用: {app.title} ({APP_VERSION})')

已加载 FastAPI 应用: ZhiShu-Agent API (0.2.0-day23)


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 测试 `GET /api/v1/health` 健康检查接口

In [2]:
health_resp = client.get('/api/v1/health')
print('状态码:', health_resp.status_code)
print('响应体:', json.dumps(health_resp.json(), ensure_ascii=False, indent=2))
assert health_resp.status_code == 200

状态码: 200
响应体: {
  "status": "ok",
  "service": "ZhiShu-Agent API",
  "version": "0.2.0-day23",
  "timestamp": "2026-11-13T05:55:55.652262+00:00"
}


## 3. 测试 `GET /api/v1/models/{model_id}`（路径参数 + 查询参数）
对比默认查询、开启 `include_pricing=true&detail_level=full`，以及查询不存在模型时的 `404` 响应。

In [3]:
r_basic = client.get('/api/v1/models/deepseek-chat')
r_full = client.get('/api/v1/models/qwen-plus?include_pricing=true&detail_level=full')
r_404 = client.get('/api/v1/models/unknown-model')

print('基础查询 (200):', r_basic.status_code, r_basic.json())
print('完整查询 (200):', r_full.status_code, r_full.json())
print('未知模型 (404):', r_404.status_code, r_404.json())

assert r_basic.status_code == 200 and r_basic.json()['pricing_per_1m_usd'] is None
assert r_full.status_code == 200 and r_full.json()['pricing_per_1m_usd'] is not None
assert r_404.status_code == 404

基础查询 (200): 200 {'model_id': 'deepseek-chat', 'provider': 'DeepSeek', 'context_window': 64000, 'supports_tools': True, 'status': 'available', 'detail_level': 'basic', 'pricing_per_1m_usd': None}
完整查询 (200): 200 {'model_id': 'qwen-plus', 'provider': 'Alibaba Cloud', 'context_window': 131072, 'supports_tools': True, 'status': 'available', 'detail_level': 'full', 'pricing_per_1m_usd': {'input': 0.4, 'output': 1.2}}
未知模型 (404): 404 {'detail': '未找到模型: unknown-model'}


## 4. 测试 `POST /api/v1/chat`（Pydantic 模型与 422 校验拦截）
验证正常对话响应（200 OK）以及空白消息、越界 `temperature`、不支持模型触发的 `422 Unprocessable Entity`。

In [4]:
chat_ok = client.post(
    '/api/v1/chat',
    json={
        'message': '请帮我总结 FastAPI 的核心优势',
        'model': 'deepseek-chat',
        'temperature': 0.6,
        'max_tokens': 256,
    },
)
print('正常对话 (200):', json.dumps(chat_ok.json(), ensure_ascii=False, indent=2))
assert chat_ok.status_code == 200

bad_cases = [
    ('空白消息', {'message': '   '}),
    ('温度越界', {'message': 'hello', 'temperature': 2.5}),
    ('未知模型', {'message': 'hello', 'model': 'invalid-model'}),
]
for label, body in bad_cases:
    resp = client.post('/api/v1/chat', json=body)
    print(f'[{label}] 状态码: {resp.status_code}, 错误详情: {resp.json()["detail"][0]["msg"]}')
    assert resp.status_code == 422

正常对话 (200): {
  "request_id": "req-986ffc0ce877",
  "session_id": "sess-71469896",
  "model": "deepseek-chat",
  "reply": "[deepseek-chat] 已收到您的请求（temperature=0.6）：请帮我总结 FastAPI 的核心优势",
  "temperature": 0.6,
  "usage": {
    "prompt_tokens": 19,
    "completion_tokens": 47,
    "total_tokens": 66
  },
  "latency_ms": 0
}
[空白消息] 状态码: 422, 错误详情: Value error, message 不能为空或仅包含空白字符
[温度越界] 状态码: 422, 错误详情: Input should be less than or equal to 2
[未知模型] 状态码: 422, 错误详情: Value error, 不支持的模型 'invalid-model'，可选模型: deepseek-chat, gpt-4o, qwen-plus
